# Toy Gaussian Experiment For Latent Drifting

This notebook implements the smallest end-to-end toy version of the proposal in [theory/proposal.typ](theory/proposal.typ):

- the true data are a 2D mixture of Gaussians with centers on a ring inside `[-5, 5]^2`
- those 2D points are embedded into `R^d` by a random isometry
- the learned models operate in the ambient `d`-dimensional space
- we inspect behavior by projecting ambient samples back onto the isometric 2D plane and separately measuring the orthogonal residual

The experiment parameters of interest are:

1. number of modes
2. ambient dimension
3. per-mode standard deviation

The architecture intentionally stays minimal:

- one time-conditioned encoder `f_phi(x, t)` with `f_phi(x, 0)` as the plain encoder
- one decoder `g_theta(z)`
- residual `Norm -> MLP -> SiLU` blocks adapted from the `ResidualMLP` idea in `../midas/prometheus`, but simplified for a notebook

Training follows the proposal at a minimal level:

- `L_dpi`: data reconstruction in ambient space
- `L_cycle^-`: Gaussian prior roundtrip
- `L_cycle^+`: data-latent roundtrip
- `L_denoise`: encoder-side latent denoising
- `L_score`: decoder-side analytic Gaussian score matching through the roundtrip trick

In [ ]:
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import einsum
from jaxtyping import Float
from pydantic import ConfigDict
from pydantic.dataclasses import dataclass


from diffusive_latent_drifting.data.toy import ToyGaussianBatch, ToyGaussianDataConfig


HEADLESS = os.environ.get("DLG_HEADLESS", "0") == "1"
SEED = int(os.environ.get("DLG_SEED", "0"))


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("seed:", SEED)


In [ ]:
@dataclass(
    kw_only=True,
    config=ConfigDict(
        arbitrary_types_allowed=True,
        extra="forbid",
    ),
)
class ExperimentConfig:
    hidden_dim: int
    projection_dim: int
    num_blocks: int
    time_dim: int
    dropout: float
    encoder_lr: float
    decoder_lr: float
    train_steps: int
    log_every: int
    warmup_steps: int
    t_min: float
    alpha_power: float
    recon_weight: float
    cycle_data_weight: float
    cycle_prior_weight: float
    denoise_weight: float
    score_weight: float
    grad_clip: float

    def __post_init__(self) -> None:
        if not (0.0 < self.t_min < 1.0):
            raise ValueError("t_min must lie in (0, 1)")


data_config = ToyGaussianDataConfig(
    seed=SEED,
    num_modes=5,
    ambient_dim=32,
    mode_std=0.35,
    ring_radius=3.0,
    plane_limit=5.0,
    embed_seed=7,
    batch_size=512,
    eval_size=2048,
    train_size=512 * 128,
    val_size=2048,
    num_workers=0,
    pin_memory=False,
    drop_last_train=True,
)

config = ExperimentConfig(
    hidden_dim=256,
    projection_dim=512,
    num_blocks=4,
    time_dim=64,
    dropout=0.0,
    encoder_lr=3e-4,
    decoder_lr=1e-4,
    train_steps=400,
    log_every=20,
    warmup_steps=100,
    t_min=0.05,
    alpha_power=1.0,
    recon_weight=10.0,
    cycle_data_weight=2.0,
    cycle_prior_weight=1.0,
    denoise_weight=1.0,
    score_weight=0.1,
    grad_clip=1.0,
)

data_config, config


In [ ]:
def project_to_plane(
    points_hd: Float[torch.Tensor, "batch ambient_dim"],
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
) -> Float[torch.Tensor, "batch plane_dim"]:
    return einsum(
        points_hd,
        basis,
        "batch ambient_dim, ambient_dim plane_dim -> batch plane_dim",
    )


def orthogonal_residual(
    points_hd: Float[torch.Tensor, "batch ambient_dim"],
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
) -> Float[torch.Tensor, "batch ambient_dim"]:
    projected = project_to_plane(points_hd, basis)
    lifted = einsum(
        projected,
        basis,
        "batch plane_dim, ambient_dim plane_dim -> batch ambient_dim",
    )
    return points_hd - lifted


def move_batch_to_device(batch: ToyGaussianBatch, device: torch.device) -> ToyGaussianBatch:
    return ToyGaussianBatch(
        x_hd=batch.x_hd.to(device),
        x_2d=batch.x_2d.to(device),
        mode_ids=batch.mode_ids.to(device),
    )


def get_next_batch(
    loader: torch.utils.data.DataLoader,
    loader_iter: iter,
) -> tuple[iter, ToyGaussianBatch]:
    try:
        batch = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        batch = next(loader_iter)
    return loader_iter, batch


def draw_noise_levels(batch_size: int, config: ExperimentConfig, device: torch.device) -> torch.Tensor:
    t = torch.rand(batch_size, 1, device=device)
    return config.t_min + (1.0 - config.t_min) * t


def gaussian_posterior_mean(y_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    one_minus_t = 1.0 - t
    coef = one_minus_t / (one_minus_t.square() + t.square())
    return coef * y_t


def noise_weight(t: torch.Tensor, power: float) -> torch.Tensor:
    return 1.0 / torch.clamp(t, min=1e-3).pow(power)


def weighted_mse(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    return ((pred - target).square() * weight).mean()


basis = data_config.get_basis().to(DEVICE)
train_loader = data_config.get_trainloader()
val_loader = data_config.get_valloader()
preview_batch = move_batch_to_device(next(iter(val_loader)), DEVICE)
preview_batch.x_hd.shape, preview_batch.x_2d.shape


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-8) -> None:
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.square().mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.scale


class ResidualMLPSiLU(nn.Module):
    '''
    Minimal adaptation of the Prometheus ResidualMLP:
    residual + normalization + MLP + SiLU, without TE/triton dependencies.
    '''

    def __init__(
        self,
        hidden_dim: int,
        projection_dim: int,
        dropout: float = 0.0,
        use_rms_norm: bool = True,
    ) -> None:
        super().__init__()
        self.norm = RMSNorm(hidden_dim) if use_rms_norm else nn.LayerNorm(hidden_dim)
        self.up = nn.Linear(hidden_dim, projection_dim)
        self.down = nn.Linear(projection_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        h = self.norm(x)
        h = F.silu(self.up(h))
        h = self.down(h)
        return residual + self.dropout(h)


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0)
            * torch.arange(half, device=t.device, dtype=t.dtype)
            / max(half - 1, 1)
        )
        angles = t * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        if emb.shape[-1] < self.dim:
            emb = F.pad(emb, (0, self.dim - emb.shape[-1]))
        return emb


class TimeConditionedResidualBlock(nn.Module):
    def __init__(
        self,
        hidden_dim: int,
        projection_dim: int,
        time_dim: int,
        dropout: float = 0.0,
        use_rms_norm: bool = True,
    ) -> None:
        super().__init__()
        self.norm = RMSNorm(hidden_dim) if use_rms_norm else nn.LayerNorm(hidden_dim)
        self.time_proj = nn.Linear(time_dim, hidden_dim)
        self.up = nn.Linear(hidden_dim, projection_dim)
        self.down = nn.Linear(projection_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor, time_features: torch.Tensor) -> torch.Tensor:
        residual = x
        h = self.norm(x)
        h = h + self.time_proj(F.silu(time_features))
        h = F.silu(self.up(h))
        h = self.down(h)
        return residual + self.dropout(h)


class Decoder(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, projection_dim: int, num_blocks: int, dropout: float) -> None:
        super().__init__()
        self.input_proj = nn.Linear(dim, hidden_dim)
        self.blocks = nn.ModuleList(
            [ResidualMLPSiLU(hidden_dim, projection_dim, dropout=dropout) for _ in range(num_blocks)]
        )
        self.out_norm = RMSNorm(hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, dim)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(z)
        for block in self.blocks:
            h = block(h)
        h = self.out_norm(h)
        return self.output_proj(h)


class TimeConditionedEncoder(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        projection_dim: int,
        time_dim: int,
        num_blocks: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, time_dim),
        )
        self.input_proj = nn.Linear(dim, hidden_dim)
        self.blocks = nn.ModuleList(
            [
                TimeConditionedResidualBlock(
                    hidden_dim=hidden_dim,
                    projection_dim=projection_dim,
                    time_dim=time_dim,
                    dropout=dropout,
                )
                for _ in range(num_blocks)
            ]
        )
        self.out_norm = RMSNorm(hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, dim)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_features = self.time_mlp(self.time_embed(t))
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, t_features)
        h = self.out_norm(h)
        return self.output_proj(h)

In [ ]:
def freeze_module(module: nn.Module, frozen: bool) -> None:
    for parameter in module.parameters():
        parameter.requires_grad_(not frozen)


def collect_metrics(
    config: ExperimentConfig,
    data_config: ToyGaussianDataConfig,
    val_loader: torch.utils.data.DataLoader,
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
    encoder: TimeConditionedEncoder,
    decoder: Decoder,
    device: torch.device,
) -> dict[str, float]:
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        batch = move_batch_to_device(next(iter(val_loader)), device)
        eval_size = int(batch.x_hd.shape[0])
        x_true = batch.x_hd
        y_true = encoder(x_true, torch.zeros(eval_size, 1, device=device))
        x_recon = decoder(y_true)
        z = torch.randn(eval_size, data_config.ambient_dim, device=device)
        x_gen = decoder(z)
        recon_mse = F.mse_loss(x_recon, x_true).item()
        recon_plane_mse = F.mse_loss(project_to_plane(x_recon, basis), batch.x_2d).item()
        recon_offplane = orthogonal_residual(x_recon, basis).norm(dim=-1).mean().item()
        gen_offplane = orthogonal_residual(x_gen, basis).norm(dim=-1).mean().item()
        gen_radius = project_to_plane(x_gen, basis).norm(dim=-1).mean().item()
    encoder.train()
    decoder.train()
    return {
        "recon_mse": recon_mse,
        "recon_plane_mse": recon_plane_mse,
        "recon_offplane": recon_offplane,
        "gen_offplane": gen_offplane,
        "gen_radius": gen_radius,
    }


def train_experiment(
    config: ExperimentConfig,
    data_config: ToyGaussianDataConfig,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
    encoder: TimeConditionedEncoder,
    decoder: Decoder,
    device: torch.device,
) -> dict[str, list[float]]:
    encoder_optimizer = torch.optim.AdamW(encoder.parameters(), lr=config.encoder_lr)
    decoder_optimizer = torch.optim.AdamW(decoder.parameters(), lr=config.decoder_lr)
    train_loader_iter = iter(train_loader)

    history: dict[str, list[float]] = {
        "step": [],
        "encoder_loss": [],
        "decoder_loss": [],
        "recon_mse": [],
        "recon_plane_mse": [],
        "recon_offplane": [],
        "gen_offplane": [],
        "gen_radius": [],
    }

    for step in range(1, config.train_steps + 1):
        train_loader_iter, batch = get_next_batch(train_loader, train_loader_iter)
        batch = move_batch_to_device(batch, device)
        x_data = batch.x_hd
        zeros = torch.zeros(x_data.shape[0], 1, device=device)

        freeze_module(decoder, frozen=True)
        freeze_module(encoder, frozen=False)
        encoder_optimizer.zero_grad(set_to_none=True)

        y_plain = encoder(x_data, zeros)
        x_recon_for_encoder = decoder(y_plain)
        y_detached = y_plain.detach()
        t = draw_noise_levels(x_data.shape[0], config, device)
        eps = torch.randn_like(y_detached)
        y_t = (1.0 - t) * y_detached + t * eps
        y_denoised = encoder(decoder(y_t), t)
        y_cycle_data = encoder(decoder(y_detached), zeros)
        z = torch.randn(x_data.shape[0], data_config.ambient_dim, device=device)
        z_cycle = encoder(decoder(z), zeros)
        alpha = noise_weight(t, config.alpha_power)

        encoder_loss = (
            config.recon_weight * F.mse_loss(x_recon_for_encoder, x_data)
            + config.cycle_data_weight * F.mse_loss(y_cycle_data, y_detached)
            + config.cycle_prior_weight * F.mse_loss(z_cycle, z)
            + config.denoise_weight * weighted_mse(y_denoised, y_detached, alpha)
        )
        encoder_loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), config.grad_clip)
        encoder_optimizer.step()

        freeze_module(decoder, frozen=False)

        train_loader_iter, batch = get_next_batch(train_loader, train_loader_iter)
        batch = move_batch_to_device(batch, device)
        x_data = batch.x_hd
        zeros = torch.zeros(x_data.shape[0], 1, device=device)

        freeze_module(encoder, frozen=True)
        freeze_module(decoder, frozen=False)
        decoder_optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            y_data = encoder(x_data, zeros)
        x_recon = decoder(y_data)
        y_roundtrip = encoder(x_recon, zeros)
        z = torch.randn(x_data.shape[0], data_config.ambient_dim, device=device)
        x_prior = decoder(z)
        z_cycle = encoder(x_prior, zeros)
        t = draw_noise_levels(x_data.shape[0], config, device)
        eps = torch.randn_like(y_roundtrip)
        y_roundtrip_t = (1.0 - t) * y_roundtrip + t * eps
        y_score = encoder(decoder(y_roundtrip_t), t)
        target = gaussian_posterior_mean(y_roundtrip_t.detach(), t)
        alpha = noise_weight(t, config.alpha_power)
        if config.warmup_steps == 0:
            score_scale = 1.0
        else:
            score_scale = min(1.0, max(0.0, (step - config.warmup_steps) / config.warmup_steps))

        decoder_loss = (
            config.recon_weight * F.mse_loss(x_recon, x_data)
            + config.cycle_data_weight * F.mse_loss(y_roundtrip, y_data)
            + config.cycle_prior_weight * F.mse_loss(z_cycle, z)
            + score_scale * config.score_weight * weighted_mse(y_score, target, alpha)
        )
        decoder_loss.backward()
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), config.grad_clip)
        decoder_optimizer.step()

        freeze_module(encoder, frozen=False)

        if step == 1 or step % config.log_every == 0 or step == config.train_steps:
            metrics = collect_metrics(
                config=config,
                data_config=data_config,
                val_loader=val_loader,
                basis=basis,
                encoder=encoder,
                decoder=decoder,
                device=device,
            )
            history["step"].append(step)
            history["encoder_loss"].append(float(encoder_loss.item()))
            history["decoder_loss"].append(float(decoder_loss.item()))
            history["recon_mse"].append(metrics["recon_mse"])
            history["recon_plane_mse"].append(metrics["recon_plane_mse"])
            history["recon_offplane"].append(metrics["recon_offplane"])
            history["gen_offplane"].append(metrics["gen_offplane"])
            history["gen_radius"].append(metrics["gen_radius"])
            print(
                f"step={step:04d} "
                f"enc={encoder_loss.item():.4f} "
                f"dec={decoder_loss.item():.4f} "
                f"recon={metrics['recon_mse']:.4f} "
                f"plane={metrics['recon_plane_mse']:.4f} "
                f"gen_off={metrics['gen_offplane']:.4f}"
            )

    return history


def make_snapshot(
    data_config: ToyGaussianDataConfig,
    val_loader: torch.utils.data.DataLoader,
    basis: Float[torch.Tensor, "ambient_dim plane_dim"],
    encoder: TimeConditionedEncoder,
    decoder: Decoder,
    device: torch.device,
) -> dict[str, torch.Tensor]:
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        batch = move_batch_to_device(next(iter(val_loader)), device)
        sample_size = int(batch.x_hd.shape[0])
        x_true_hd = batch.x_hd
        x_true_2d = batch.x_2d
        y_true = encoder(x_true_hd, torch.zeros(sample_size, 1, device=device))
        x_recon_hd = decoder(y_true)
        z = torch.randn(sample_size, data_config.ambient_dim, device=device)
        x_gen_hd = decoder(z)
    encoder.train()
    decoder.train()
    return {
        "x_true_2d": x_true_2d.detach().cpu(),
        "x_true_hd": x_true_hd.detach().cpu(),
        "x_recon_2d": project_to_plane(x_recon_hd, basis).detach().cpu(),
        "x_recon_hd": x_recon_hd.detach().cpu(),
        "x_gen_2d": project_to_plane(x_gen_hd, basis).detach().cpu(),
        "x_gen_hd": x_gen_hd.detach().cpu(),
        "recon_offplane": orthogonal_residual(x_recon_hd, basis).norm(dim=-1).detach().cpu(),
        "gen_offplane": orthogonal_residual(x_gen_hd, basis).norm(dim=-1).detach().cpu(),
    }


def show_or_close(fig: plt.Figure) -> None:
    if HEADLESS:
        plt.close(fig)
    else:
        plt.show()


def plot_training_curves(history: dict[str, list[float]]) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history["step"], history["encoder_loss"], label="encoder")
    axes[0].plot(history["step"], history["decoder_loss"], label="decoder")
    axes[0].set_title("Optimization Losses")
    axes[0].set_xlabel("step")
    axes[0].legend()

    axes[1].plot(history["step"], history["recon_mse"], label="ambient recon mse")
    axes[1].plot(history["step"], history["recon_plane_mse"], label="plane recon mse")
    axes[1].set_title("Reconstruction")
    axes[1].set_xlabel("step")
    axes[1].legend()

    axes[2].plot(history["step"], history["recon_offplane"], label="recon off-plane")
    axes[2].plot(history["step"], history["gen_offplane"], label="gen off-plane")
    axes[2].set_title("Orthogonal Residual")
    axes[2].set_xlabel("step")
    axes[2].legend()

    fig.tight_layout()
    show_or_close(fig)


def plot_snapshot(data_config: ToyGaussianDataConfig, snapshot: dict[str, torch.Tensor]) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    plane_limit = data_config.plane_limit

    axes[0, 0].scatter(
        snapshot["x_true_2d"][:, 0],
        snapshot["x_true_2d"][:, 1],
        s=8,
        alpha=0.35,
    )
    axes[0, 0].set_title("True Data In 2D")

    recon = axes[0, 1].scatter(
        snapshot["x_recon_2d"][:, 0],
        snapshot["x_recon_2d"][:, 1],
        c=snapshot["recon_offplane"],
        s=8,
        alpha=0.45,
        cmap="viridis",
    )
    axes[0, 1].set_title("Reconstructions, colored by off-plane norm")
    fig.colorbar(recon, ax=axes[0, 1], fraction=0.046, pad=0.04)

    gen = axes[1, 0].scatter(
        snapshot["x_gen_2d"][:, 0],
        snapshot["x_gen_2d"][:, 1],
        c=snapshot["gen_offplane"],
        s=8,
        alpha=0.45,
        cmap="magma",
    )
    axes[1, 0].set_title("Generated Samples, colored by off-plane norm")
    fig.colorbar(gen, ax=axes[1, 0], fraction=0.046, pad=0.04)

    axes[1, 1].hist(snapshot["recon_offplane"].numpy(), bins=40, alpha=0.65, label="recon")
    axes[1, 1].hist(snapshot["gen_offplane"].numpy(), bins=40, alpha=0.65, label="generated")
    axes[1, 1].set_title("Orthogonal Residual Histogram")
    axes[1, 1].legend()

    for axis in [axes[0, 0], axes[0, 1], axes[1, 0]]:
        axis.set_xlim(-plane_limit, plane_limit)
        axis.set_ylim(-plane_limit, plane_limit)
        axis.set_aspect("equal")

    fig.tight_layout()
    show_or_close(fig)


def build_models(
    config: ExperimentConfig,
    data_config: ToyGaussianDataConfig,
    device: torch.device,
) -> tuple[TimeConditionedEncoder, Decoder]:
    encoder = TimeConditionedEncoder(
        dim=data_config.ambient_dim,
        hidden_dim=config.hidden_dim,
        projection_dim=config.projection_dim,
        time_dim=config.time_dim,
        num_blocks=config.num_blocks,
        dropout=config.dropout,
    ).to(device)
    decoder = Decoder(
        dim=data_config.ambient_dim,
        hidden_dim=config.hidden_dim,
        projection_dim=config.projection_dim,
        num_blocks=config.num_blocks,
        dropout=config.dropout,
    ).to(device)
    return encoder, decoder


def run_single_experiment(
    config: ExperimentConfig,
    data_config: ToyGaussianDataConfig,
) -> dict[str, object]:
    set_seed(SEED)
    basis = data_config.get_basis().to(DEVICE)
    train_loader = data_config.get_trainloader()
    val_loader = data_config.get_valloader()
    encoder, decoder = build_models(config, data_config, DEVICE)
    history = train_experiment(
        config=config,
        data_config=data_config,
        train_loader=train_loader,
        val_loader=val_loader,
        basis=basis,
        encoder=encoder,
        decoder=decoder,
        device=DEVICE,
    )
    snapshot = make_snapshot(
        data_config=data_config,
        val_loader=val_loader,
        basis=basis,
        encoder=encoder,
        decoder=decoder,
        device=DEVICE,
    )
    metrics = collect_metrics(
        config=config,
        data_config=data_config,
        val_loader=val_loader,
        basis=basis,
        encoder=encoder,
        decoder=decoder,
        device=DEVICE,
    )
    return {
        "config": config,
        "data_config": data_config,
        "basis": basis.detach().cpu(),
        "encoder": encoder,
        "decoder": decoder,
        "history": history,
        "snapshot": snapshot,
        "metrics": metrics,
    }


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(preview_batch.x_2d[:, 0].cpu(), preview_batch.x_2d[:, 1].cpu(), s=8, alpha=0.35)
ax.set_xlim(-data_config.plane_limit, data_config.plane_limit)
ax.set_ylim(-data_config.plane_limit, data_config.plane_limit)
ax.set_aspect("equal")
ax.set_title(
    f"True 2D mixture: modes={data_config.num_modes}, ambient_dim={data_config.ambient_dim}, std={data_config.mode_std}"
)
fig.tight_layout()
show_or_close(fig)


In [ ]:
results = run_single_experiment(config=config, data_config=data_config)
print(results["metrics"])


In [ ]:
plot_training_curves(results["history"])
plot_snapshot(results["data_config"], results["snapshot"])


## Notes for sweeps

The three main scientific knobs live on `data_config`:

- `num_modes`
- `ambient_dim`
- `mode_std`

`config` is reserved for model and optimization settings.

For sweeps, edit the literal `ToyGaussianDataConfig(...)` instantiation above and keep `config` fixed unless you explicitly want to retune optimization or architecture.

The two core diagnostics to watch are:

- the projected 2D distribution after decoding
- the orthogonal residual norm, which isolates ambient-space behavior outside the informative 2D isometric subspace
